# 5.2. 参数管理

在选择了架构并设置了超参数后，我们就进入了训练阶段。此时，我们的目标是找到使损失函数最小化的模型参数值。经过训练后，我们将需要使用这些参数来做出未来的预测。

In [ ]:
import tensorflow as tf
import numpy as np

# 创建一个简单的Sequential网络
net = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(4, activation=tf.nn.relu),
    tf.keras.layers.Dense(1),
])

X = tf.random.uniform((2, 4))
net(X)  # 前向传播以初始化参数

## 5.2.1. 参数访问

In [ ]:
# 访问第二层（索引为1）的参数
print(net.layers[2].weights)

In [ ]:
# 访问目标参数
print(type(net.layers[2].weights[1]), net.layers[2].weights[1])

In [ ]:
# 访问参数值
print(net.layers[2].weights[1])
print(tf.convert_to_tensor(net.layers[2].weights[1]))

### 5.2.1.1. 一次性访问所有参数

In [ ]:
# 访问所有层的参数
print(net.get_weights())

In [ ]:
# 访问第一个全连接层的参数
print(net.layers[1].weights)
print(net.layers[1].weights[0])

### 5.2.1.2. 从嵌套块收集参数

In [ ]:
# 定义一个嵌套块
def block1():
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation=tf.nn.relu)
    ])

def block2():
    net = tf.keras.Sequential()
    for i in range(4):
        net.add(block1())
    return net

rgnet = tf.keras.Sequential()
rgnet.add(block2())
rgnet.add(tf.keras.layers.Dense(10))
rgnet(X)

In [ ]:
# 打印网络结构
print(rgnet.summary())

In [ ]:
# 访问特定层的参数
rgnet.layers[0].layers[1].layers[1].weights

## 5.2.2. 参数初始化

In [ ]:
# 默认情况下，Keras使用Glorot均匀初始化器
# 我们可以使用不同的初始化器
net = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(
        4, activation=tf.nn.relu,
        kernel_initializer=tf.random_normal_initializer(mean=0, stddev=0.01),
        bias_initializer=tf.zeros_initializer()),
    tf.keras.layers.Dense(1)
])

net(X)
print(net.layers[1].weights[0])
print(net.layers[1].weights[1])

### 5.2.2.1. 内置初始化

In [ ]:
# 常数初始化
net = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(
        4, activation=tf.nn.relu,
        kernel_initializer=tf.keras.initializers.Constant(1),
        bias_initializer=tf.zeros_initializer()),
    tf.keras.layers.Dense(1),
])

net(X)
print(net.layers[1].weights[0])
print(net.layers[1].weights[1])

In [ ]:
# Xavier初始化
net = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(
        4,
        activation=tf.nn.relu,
        kernel_initializer=tf.keras.initializers.GlorotUniform()),
    tf.keras.layers.Dense(
        1, kernel_initializer=tf.keras.initializers.GlorotUniform()),
])

net(X)
print(net.layers[1].weights[0])

### 5.2.2.2. 自定义初始化

In [ ]:
# 自定义初始化器
class MyInit(tf.keras.initializers.Initializer):
    def __call__(self, shape, dtype=None):
        data = tf.random.uniform(shape, -10, 10, dtype=dtype)
        factor = tf.cast(tf.abs(data) >= 5, dtype=dtype)
        return data * factor

net = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(
        4,
        activation=tf.nn.relu,
        kernel_initializer=MyInit()),
    tf.keras.layers.Dense(1),
])

net(X)
print(net.layers[1].weights[0])

In [ ]:
# 直接设置参数
net.layers[1].weights[0][:].assign(net.layers[1].weights[0] + 1)
net.layers[1].weights[0][0, 0].assign(42)
print(net.layers[1].weights[0])

## 5.2.3. 参数绑定

In [ ]:
# 创建共享层
shared = tf.keras.layers.Dense(4, activation=tf.nn.relu)
net = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    shared,
    shared,
    tf.keras.layers.Dense(1),
])

net(X)
# 检查参数是否相同
print(len(net.layers) == 4)
print(net.layers[1].weights[0] is net.layers[2].weights[0])

## 小结

- 我们有几种方法可以访问、初始化和绑定模型参数。
- 我们可以使用自定义初始化方法。